In [ ]:
# Add target col

import pandas as pd
import nba_api_module as nbpim

df_input = pd.read_csv("data/ml_ready_2024_25.csv")

# Drop star injury
df_input.drop(columns=['home_missing_starters', 'away_missing_starters'], inplace=True)


gid = f"00{df_input.loc[0, "game_id"]}"

season_log = nbpim.get_season_game_log("2024-25")
season_log_home = season_log[season_log['MATCHUP'].str.contains(' vs. ')]
season_log_home['is_home_win'] = season_log_home['WL'].apply(lambda x: 1 if x == 'W' else 0)
season_log_home.rename(columns={'GAME_ID': 'game_id'}, inplace=True)
season_log_home.game_id = season_log_home.game_id.astype(int)

df_input = pd.merge(df_input, season_log_home[['game_id', 'is_home_win']],
                    on="game_id", how="left")

print(df_input.sample(5))
print(df_input.columns)

In [ ]:
# Feature engineering

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------------------
# 1) DIFFERENCIÁLIS FEATURES HOZZÁADÁSA
# -------------------------------------------------------------------------

df = df_input.copy()

# Offensive/Defensive Rating különbségek
df['ORtg_diff'] = df['home_ORtg'] - df['away_ORtg']
df['DRtg_diff'] = df['home_DRtg'] - df['away_DRtg']
df['NET_rtg_diff'] = df['home_NET_rtg'] - df['away_NET_rtg']

# Pace különbség
df['PACE_diff'] = df['home_PACE'] - df['away_PACE']

# Hatékonyság különbségek
df['TS_diff'] = df['home_TS%'] - df['away_TS%']
df['EFG_diff'] = df['home_EFG%'] - df['away_EFG%']
df['AST_ratio_diff'] = df['home_AST_ratio'] - df['away_AST_ratio']
df['OREB_diff'] = df['home_OREB%'] - df['away_OREB%']
df['turnover_diff'] = df['home_turnover_ratio'] - df['away_turnover_ratio']

# Játékos minőség különbségek
df['starter_PER_diff'] = df['home_starter_avg_PER'] - df['away_starter_avg_PER']
df['bench_PER_diff'] = df['home_bench_avg_PER'] - df['away_bench_avg_PER']
df['star_usage_diff'] = df['home_star_usage'] - df['away_star_usage']
df['avg_TS_diff'] = df['home_avg_TS'] - df['away_avg_TS']
df['top3_points_diff'] = df['home_top3_points_avg'] - df['away_top3_points_avg']

# Pihenés és forma különbségek
df['rest_days_diff'] = df['home_rest_days'] - df['away_rest_days']
df['recent_form10_diff'] = df['home_recent_form10'] - df['away_recent_form10']
df['recent_form5_diff'] = df['home_recent_form5'] - df['away_recent_form5']
df['recent_form3_diff'] = df['home_recent_form3'] - df['away_recent_form3']

# Sérülés különbség
df['injury_count_diff'] = df['home_injury_count'] - df['away_injury_count']

# Back-to-back advantage (1 ha csak away, -1 ha csak home, 0 ha mindkettő vagy egyik sem)
df['b2b_advantage'] = df['away_is_back_to_back'].astype(int) - df['home_is_back_to_back'].astype(int)

print(f"Új differenciális features hozzáadva. Új shape: {df.shape}")
print(f"\nÚj feature-ök száma: {len([col for col in df.columns if 'diff' in col or 'advantage' in col])}")

# -------------------------------------------------------------------------
# 2) CORRELATION VIZSGÁLAT - CSAK NUMERIKUS FEATURE-ÖK
# -------------------------------------------------------------------------

# Exclude target és ID
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'is_home_win' in numeric_cols:
    numeric_cols.remove('is_home_win')
if 'game_id' in numeric_cols:
    numeric_cols.remove('game_id')

# Correlation matrix
corr_matrix = df[numeric_cols].corr()

# Erősen korreláló párok keresése (|corr| > 0.85, de nem önmagával)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.85:
            high_corr_pairs.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

print(f"\n=== ERŐSEN KORRELÁLÓ FEATURE PÁROK (|corr| > 0.85) ===\n")
if high_corr_pairs:
    corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', 
                                                          key=abs, 
                                                          ascending=False)
    print(corr_df.to_string(index=False))
else:
    print("Nincs erősen korreláló feature pár (|corr| > 0.85)")

# -------------------------------------------------------------------------
# 3) CORRELATION HEATMAP (TOP 30 FEATURE TARGET-TEL)
# -------------------------------------------------------------------------

# Correlation a target változóval
target_corr = df[numeric_cols].corrwith(df['is_home_win']).abs().sort_values(ascending=False)

print(f"\n=== TOP 15 FEATURE CORRELATION A TARGET-TEL ===\n")
print(target_corr.head(15))

# Vizualizáció: top 30 feature heatmap
top_features = target_corr.head(30).index.tolist()
top_features.append('is_home_win')

plt.figure(figsize=(16, 14))
sns.heatmap(df[top_features].corr(), 
            annot=False, 
            cmap='coolwarm', 
            center=0,
            vmin=-1, vmax=1,
            square=True,
            linewidths=0.5)
plt.title('Correlation Heatmap - Top 30 Features + Target', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------------
# 4) AJÁNLÁSOK REDUNDÁNS FEATURES-ÖK ELTÁVOLÍTÁSÁRA
# -------------------------------------------------------------------------

print(f"\n=== AJÁNLÁS: REDUNDÁNS FEATURES ELTÁVOLÍTÁSA ===\n")

features_to_remove = []
for pair in high_corr_pairs:
    feat1, feat2 = pair['Feature 1'], pair['Feature 2']
    # Tartsd meg azt amelyik jobban korrelál a target-tel
    corr1 = abs(df[feat1].corr(df['is_home_win']))
    corr2 = abs(df[feat2].corr(df['is_home_win']))
    
    if corr1 > corr2:
        features_to_remove.append(feat2)
    else:
        features_to_remove.append(feat1)

features_to_remove = list(set(features_to_remove))

if features_to_remove:
    print(f"Javasolt eltávolítandó features ({len(features_to_remove)} db):")
    for feat in features_to_remove:
        print(f"  - {feat}")
else:
    print("Nincs javasolt eltávolítandó feature.")

# -------------------------------------------------------------------------
# 5) UPDATED DATAFRAME EXPORT
# -------------------------------------------------------------------------

print(f"\n=== EREDMÉNY ===")
print(f"Eredeti feature-ök száma: {len(df_input.columns)}")
print(f"Új differenciális features: {len([col for col in df.columns if 'diff' in col or 'advantage' in col])}")
print(f"Összes feature most: {len(df.columns)}")
print(f"\nHasználd a 'df' dataframe-et a továbbiakban a bővített feature-ökkel!")

# Használható változó
df_enhanced = df.copy()

In [ ]:
# ML teszt

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------
# 1) ADATOK ELŐKÉSZÍTÉSE + ODDS MERGE
# -------------------------------------------------------------------------

df = df_enhanced.copy()
df.dropna(subset=["is_home_win"], inplace=True)

# Odds merge (feltételezzük hogy odds df-ben van GAME_ID, odds_home, odds_away)
odds = pd.read_csv("data/game_odds_2024_25.csv")
df = df.merge(odds[['GAME_ID', 'odds_home', 'odds_away']], 
              left_on='game_id', right_on='GAME_ID', how='left')
df.drop(columns=['GAME_ID'], inplace=True)

# Implied probabilities számítása
df['implied_prob_home'] = 1 / df['odds_home']
df['implied_prob_away'] = 1 / df['odds_away']

# Margin (bookmaker edge)
df['book_margin'] = df['implied_prob_home'] + df['implied_prob_away'] - 1

print(f"Dataset méret odds-szal: {df.shape}")
print(f"Odds hiányzó értékek: {df['odds_home'].isna().sum()}")

# Célváltozó
y = df["is_home_win"]

# Features (game_id, odds és target nélkül)
feature_cols = [col for col in df.columns if col not in 
                ['is_home_win', 'game_id', 'odds_home', 'odds_away', 
                 'implied_prob_home', 'implied_prob_away', 'book_margin']]
X = df[feature_cols]

print(f"Features száma: {X.shape[1]}")

# -------------------------------------------------------------------------
# 2) PCA VIZSGÁLAT
# -------------------------------------------------------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCA()
pca_full.fit(X_scaled)

cumsum_variance = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = np.argmax(cumsum_variance >= 0.95) + 1
n_components_90 = np.argmax(cumsum_variance >= 0.90) + 1

print(f"\n=== PCA VARIANCE ANALÍZIS ===")
print(f"95% variance: {n_components_95}/{X.shape[1]} komponens")
print(f"90% variance: {n_components_90}/{X.shape[1]} komponens")

# -------------------------------------------------------------------------
# 3) TIME-SERIES CV + BETTING EVALUATION
# -------------------------------------------------------------------------

def time_series_cv_split(X, y, n_splits=5, test_size=0.2):
    n_samples = len(X)
    test_size_samples = int(n_samples * test_size)
    step = test_size_samples
    
    splits = []
    for i in range(n_splits):
        test_end = n_samples - (n_splits - 1 - i) * step
        test_start = test_end - test_size_samples
        train_end = test_start
        
        if train_end < int(n_samples * 0.3):
            continue
            
        train_idx = np.arange(0, train_end)
        test_idx = np.arange(test_start, test_end)
        splits.append((train_idx, test_idx))
    
    return splits

def calculate_betting_metrics(y_true, pred_proba, odds_home, odds_away, 
                               threshold=0.00, unit_stake=1.0):
    """
    Betting szimuláció metrikákkal
    
    Args:
        threshold: Minimális predicted probability betting-hez
        unit_stake: Egységnyi tét (default 1.0)
    """
    bets = []
    cumulative_profit = 0
    peak_balance = 0
    max_drawdown = 0
    
    for i in range(len(y_true)):
        pred_home_prob = pred_proba[i]
        pred_away_prob = 1 - pred_home_prob
        
        actual_outcome = y_true.iloc[i]
        home_odds = odds_home.iloc[i]
        away_odds = odds_away.iloc[i]
        
        # Bet decision
        bet_placed = False
        bet_on_home = False
        stake = 0
        profit = 0
        
        # Bet on HOME if model confidence > threshold
        if (pred_home_prob >= threshold) & (pred_home_prob > 1/home_odds):
            bet_placed = True
            bet_on_home = True
            stake = unit_stake
            
            if actual_outcome == 1:  # Home wins
                profit = stake * (home_odds - 1)
            else:
                profit = -stake
        
        # Bet on AWAY if model confidence > threshold
        elif (pred_away_prob >= threshold) & (pred_away_prob > 1/away_odds):
            bet_placed = True
            bet_on_home = False
            stake = unit_stake
            
            if actual_outcome == 0:  # Away wins
                profit = stake * (away_odds - 1)
            else:
                profit = -stake
        
        if bet_placed:
            cumulative_profit += profit
            bets.append({
                'bet_on_home': bet_on_home,
                'stake': stake,
                'profit': profit,
                'cumulative': cumulative_profit,
                'odds': home_odds if bet_on_home else away_odds,
                'won': profit > 0
            })
            
            # Drawdown calculation
            if cumulative_profit > peak_balance:
                peak_balance = cumulative_profit
            
            drawdown = peak_balance - cumulative_profit
            if drawdown > max_drawdown:
                max_drawdown = drawdown
    
    if len(bets) == 0:
        return {
            'total_bets': 0,
            'total_profit': 0,
            'roi': 0,
            'hit_rate': 0,
            'avg_odds': 0,
            'max_drawdown': 0,
            'ev_per_bet': 0
        }
    
    bets_df = pd.DataFrame(bets)
    total_stake = bets_df['stake'].sum()
    total_profit = bets_df['profit'].sum()
    
    return {
        'total_bets': len(bets),
        'total_profit': total_profit,
        'roi': (total_profit / total_stake * 100) if total_stake > 0 else 0,
        'hit_rate': (bets_df['won'].sum() / len(bets) * 100),
        'avg_odds': bets_df['odds'].mean(),
        'max_drawdown': max_drawdown,
        'ev_per_bet': total_profit / len(bets),
        'bets_details': bets_df
    }

def evaluate_model_cv(model, X, y, df_full, cv_splits, use_pca=False, n_components=None):
    """
    Modell kiértékelése ML és Betting metrikákkal
    """
    # ML metrics
    accuracies = []
    aucs = []
    log_losses = []
    
    # Betting metrics
    all_betting_metrics = []
    
    for train_idx, test_idx in cv_splits:
        X_train_fold = X[train_idx]
        X_test_fold = X[test_idx]
        y_train_fold = y.iloc[train_idx]
        y_test_fold = y.iloc[test_idx]
        
        # Odds for test fold
        odds_home_test = df_full['odds_home'].iloc[test_idx]
        odds_away_test = df_full['odds_away'].iloc[test_idx]
        
        # Scaling
        scaler_fold = StandardScaler()
        X_train_scaled = scaler_fold.fit_transform(X_train_fold)
        X_test_scaled = scaler_fold.transform(X_test_fold)
        
        # PCA if needed
        if use_pca and n_components:
            pca = PCA(n_components=n_components)
            X_train_final = pca.fit_transform(X_train_scaled)
            X_test_final = pca.transform(X_test_scaled)
        else:
            if "Logistic" in str(type(model).__name__):
                X_train_final = X_train_scaled
                X_test_final = X_test_scaled
            else:
                X_train_final = X_train_fold
                X_test_final = X_test_fold
        
        # Training
        model.fit(X_train_final, y_train_fold)
        
        # Prediction
        pred_proba = model.predict_proba(X_test_final)[:, 1]
        preds = (pred_proba > 0.5).astype(int)
        
        # ML Metrics
        accuracies.append(accuracy_score(y_test_fold, preds))
        aucs.append(roc_auc_score(y_test_fold, pred_proba))
        log_losses.append(log_loss(y_test_fold, pred_proba))
        
        # Betting Metrics
        betting_metrics = calculate_betting_metrics(
            y_test_fold, pred_proba, odds_home_test, odds_away_test
        )
        all_betting_metrics.append(betting_metrics)
    
    # Aggregate betting metrics
    total_bets = sum([m['total_bets'] for m in all_betting_metrics])
    total_profit = sum([m['total_profit'] for m in all_betting_metrics])
    
    if total_bets > 0:
        avg_roi = np.mean([m['roi'] for m in all_betting_metrics if m['total_bets'] > 0])
        avg_hit_rate = np.mean([m['hit_rate'] for m in all_betting_metrics if m['total_bets'] > 0])
        avg_ev = np.mean([m['ev_per_bet'] for m in all_betting_metrics if m['total_bets'] > 0])
        max_dd = np.max([m['max_drawdown'] for m in all_betting_metrics])
    else:
        avg_roi = avg_hit_rate = avg_ev = max_dd = 0
    
    return {
        # ML Metrics
        "Accuracy_mean": np.mean(accuracies),
        "Accuracy_std": np.std(accuracies),
        "AUC_mean": np.mean(aucs),
        "AUC_std": np.std(aucs),
        "LogLoss_mean": np.mean(log_losses),
        "LogLoss_std": np.std(log_losses),
        
        # Betting Metrics
        "Total_Bets": total_bets,
        "Total_Profit": total_profit,
        "ROI_%": avg_roi,
        "Hit_Rate_%": avg_hit_rate,
        "EV_per_Bet": avg_ev,
        "Max_Drawdown": max_dd
    }

# -------------------------------------------------------------------------
# 4) MODELLEK
# -------------------------------------------------------------------------

xgb_tuned = XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    min_child_weight=5, gamma=1.0, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
    eval_metric="logloss", use_label_encoder=False, random_state=42
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost (Tuned)": xgb_tuned
}

models_pca = {
    "Logistic Regression + PCA": LogisticRegression(max_iter=500, random_state=42),
    "Random Forest + PCA": RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42),
    "XGBoost (Tuned) + PCA": XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        min_child_weight=5, gamma=1.0, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
        eval_metric="logloss", use_label_encoder=False, random_state=42
    ),
    "XGBoost (Very Shallow) + PCA": XGBClassifier(
        n_estimators=50, learning_rate=0.05, max_depth=2,
        min_child_weight=20, gamma=3.0, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=2.0, reg_lambda=10.0,
        eval_metric="logloss", use_label_encoder=False, random_state=42
    ),

}

# -------------------------------------------------------------------------
# 5) KIÉRTÉKELÉS
# -------------------------------------------------------------------------

cv_splits = time_series_cv_split(X, y, n_splits=5)

print(f"\n{'='*60}")
print("=== MODELLEK KIÉRTÉKELÉSE (ML + BETTING) ===")
print(f"{'='*60}\n")

results = {}

print("--- Standard Features ---\n")
for name, model in models.items():
    print(f"Futtatás: {name}...")
    results[name] = evaluate_model_cv(model, X.values, y, df, cv_splits, use_pca=False)

print("\n--- PCA Features (90% variance) ---\n")
for name, model in models_pca.items():
    print(f"Futtatás: {name}...")
    results[name] = evaluate_model_cv(model, X.values, y, df, cv_splits, 
                                       use_pca=True, n_components=n_components_90)

# -------------------------------------------------------------------------
# 6) EREDMÉNYEK
# -------------------------------------------------------------------------

results_df = pd.DataFrame(results).T

print(f"\n{'='*70}")
print("=== ML METRICS ===")
print(f"{'='*70}\n")
ml_cols = ['Accuracy_mean', 'Accuracy_std', 'AUC_mean', 'AUC_std', 'LogLoss_mean', 'LogLoss_std']
print(results_df[ml_cols].sort_values('AUC_mean', ascending=False).to_string())

print(f"\n{'='*70}")
print("=== BETTING METRICS ===")
print(f"{'='*70}\n")
bet_cols = ['Total_Bets', 'Total_Profit', 'ROI_%', 'Hit_Rate_%', 'EV_per_Bet', 'Max_Drawdown']
betting_results = results_df[bet_cols].sort_values('ROI_%', ascending=False)
print(betting_results.to_string())

# Best models
best_ml = results_df['AUC_mean'].idxmax()
best_betting = results_df['ROI_%'].idxmax()

print(f"\n{'='*70}")
print(f"🏆 LEGJOBB ML MODELL: {best_ml}")
print(f"   AUC: {results_df.loc[best_ml, 'AUC_mean']:.4f}")
print(f"\n💰 LEGJOBB BETTING MODELL: {best_betting}")
print(f"   ROI: {results_df.loc[best_betting, 'ROI_%']:.2f}%")
print(f"   Total Profit: {results_df.loc[best_betting, 'Total_Profit']:.2f} units")
print(f"   Hit Rate: {results_df.loc[best_betting, 'Hit_Rate_%']:.2f}%")
print(f"   EV/Bet: {results_df.loc[best_betting, 'EV_per_Bet']:.3f} units")
print(f"{'='*70}")

# -------------------------------------------------------------------------
# 7) VIZUALIZÁCIÓ
# -------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# AUC
results_df.sort_values('AUC_mean')['AUC_mean'].plot(
    kind='barh', xerr=results_df.sort_values('AUC_mean')['AUC_std'],
    ax=axes[0,0], color='skyblue', edgecolor='black'
)
axes[0,0].set_xlabel('AUC')
axes[0,0].set_title('Model AUC Scores')
axes[0,0].axvline(x=0.7, color='red', linestyle='--', alpha=0.5)

# ROI
results_df.sort_values('ROI_%')['ROI_%'].plot(
    kind='barh', ax=axes[0,1], color='green', edgecolor='black'
)
axes[0,1].set_xlabel('ROI (%)')
axes[0,1].set_title('Betting ROI')
axes[0,1].axvline(x=0, color='red', linestyle='--', alpha=0.5)

# Hit Rate
results_df.sort_values('Hit_Rate_%')['Hit_Rate_%'].plot(
    kind='barh', ax=axes[1,0], color='orange', edgecolor='black'
)
axes[1,0].set_xlabel('Hit Rate (%)')
axes[1,0].set_title('Betting Hit Rate')
axes[1,0].axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% baseline')

# Total Profit
results_df.sort_values('Total_Profit')['Total_Profit'].plot(
    kind='barh', ax=axes[1,1], color='purple', edgecolor='black'
)
axes[1,1].set_xlabel('Total Profit (units)')
axes[1,1].set_title('Betting Total Profit')
axes[1,1].axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# XGBoost + PCA deep dive

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

# -------------------------------------------------------------------------
# 1) MODELL TRAINING ÉS PREDIKCIÓK GYŰJTÉSE
# -------------------------------------------------------------------------

print("="*70)
print("=== XGBOOST + PCA MODELL DEEP DIVE ANALÍZIS ===")
print("="*70)

df = df_enhanced.copy()
df.dropna(subset=["is_home_win"], inplace=True)

# Odds merge
df = df.merge(odds[['GAME_ID', 'odds_home', 'odds_away']], 
              left_on='game_id', right_on='GAME_ID', how='left')
df.drop(columns=['GAME_ID'], inplace=True)

df['implied_prob_home'] = 1 / df['odds_home']
df['implied_prob_away'] = 1 / df['odds_away']

y = df["is_home_win"]
feature_cols = [col for col in df.columns if col not in 
                ['is_home_win', 'game_id', 'odds_home', 'odds_away', 
                 'implied_prob_home', 'implied_prob_away', 'book_margin']]
X = df[feature_cols]

# Time-series split
split_idx = int(len(df) * 0.8)
X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

# Scaling + PCA
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=0.90)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"\nPCA Components: {pca.n_components_} (90% variance)")

# Model training
cfg = {"name": "Very Shallow", "max_depth": 2, "min_child_weight": 20, 
        "gamma": 3.0, "reg_alpha": 2.0, "reg_lambda": 10.0, "n_estimators": 50}
print(f"Config: {cfg['name']}")

model_name = 'Random Forest + PCA'
print(model_name)
xgb_model = models_pca[model_name]
xgb_model.fit(X_train_pca, y_train)

# Predictions
train_proba = xgb_model.predict_proba(X_train_pca)[:, 1]
test_proba = xgb_model.predict_proba(X_test_pca)[:, 1]

# Create results dataframe
test_results = df.iloc[split_idx:].copy()
test_results['pred_prob_home'] = test_proba
test_results['pred_prob_away'] = 1 - test_proba

# -------------------------------------------------------------------------
# 2) PCA KOMPONENS ANALÍZIS
# -------------------------------------------------------------------------

print(f"\n{'='*70}")
print("=== PCA KOMPONENS ELEMZÉS ===")
print(f"{'='*70}\n")

# Top 5 legfontosabb komponens
print(f"Első 5 komponens variance magyarázata:")
for i in range(min(5, pca.n_components_)):
    print(f"  PC{i+1}: {pca.explained_variance_ratio_[i]*100:.2f}%")

print(f"\nÖsszesen {pca.n_components_} komponens használva")
print(f"Teljes variance: {pca.explained_variance_ratio_.sum()*100:.2f}%")

# Top features az első 3 komponensben
print(f"\n--- Top 5 Feature Loadings (első 3 PC) ---")
components_df = pd.DataFrame(
    pca.components_[:3],
    columns=X.columns,
    index=[f'PC{i+1}' for i in range(3)]
)

for pc_name, pc_values in components_df.iterrows():
    top_features = pc_values.abs().nlargest(5)
    print(f"\n{pc_name}:")
    for feat, val in top_features.items():
        print(f"  {feat}: {val:.3f}")

# -------------------------------------------------------------------------
# 3) KALIBRÁCIÓ VIZSGÁLAT
# -------------------------------------------------------------------------

print(f"\n{'='*70}")
print("=== MODELL KALIBRÁCIÓ ===")
print(f"{'='*70}\n")

# Brier score
train_brier = brier_score_loss(y_train, train_proba)
test_brier = brier_score_loss(y_test, test_proba)

print(f"Brier Score (Train): {train_brier:.4f}")
print(f"Brier Score (Test):  {test_brier:.4f}")
print(f"Delta: {abs(test_brier - train_brier):.4f} {'(overfitting)' if test_brier > train_brier + 0.01 else '(jó generalizáció)'}")

# Calibration curve
prob_true, prob_pred = calibration_curve(y_test, test_proba, n_bins=10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration plot
axes[0].plot([0, 1], [0, 1], 'k--', label='Tökéletesen kalibrált')
axes[0].plot(prob_pred, prob_true, 'o-', linewidth=2, label='XGBoost+PCA')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('True Probability')
axes[0].set_title('Calibration Curve (Test Set)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Probability distribution
axes[1].hist(test_proba, bins=30, alpha=0.7, edgecolor='black')
axes[1].axvline(0.50, color='red', linestyle='--', linewidth=2, label='0.50')
axes[1].set_xlabel('Predicted Probability (Home Win)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Prediction Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 5) ODDS RANGE ANALÍZIS
# -------------------------------------------------------------------------

print(f"\n{'='*70}")
print("=== ODDS TARTOMÁNY TELJESÍTMÉNY ===")
print(f"{'='*70}\n")

# Odds bins
test_results['odds_bin'] = pd.cut(
    test_results.apply(lambda x: x['odds_home'] if x['pred_prob_home'] >= 1/x['odds_home'] 
                       else x['odds_away'] if x['pred_prob_away'] >= 1/x['odds_away']
                       else 0, axis=1),
    bins=[1.0, 1.5, 2.0, 2.5, 3.0, 10.0],
    labels=['1.0-1.5 (Heavy Fav)', '1.5-2.0 (Favorite)', 
            '2.0-2.5 (Slight Fav)', '2.5-3.0 (Underdog)', '3.0+ (Heavy Dog)']
)

odds_analysis = []
for odds_range in test_results['odds_bin'].dropna().unique():
    subset = test_results[test_results['odds_bin'] == odds_range]
    
    bets = []
    for idx, row in subset.iterrows():
        if row['pred_prob_home'] >= 1/row['odds_home']:
            profit = (row['odds_home'] - 1) if row['is_home_win'] == 1 else -1
            bets.append(profit)
        elif row['pred_prob_away'] >= 1/row['odds_away']:
            profit = (row['odds_away'] - 1) if row['is_home_win'] == 0 else -1
            bets.append(profit)
    
    if len(bets) > 0:
        odds_analysis.append({
            'odds_range': odds_range,
            'num_bets': len(bets),
            'total_profit': sum(bets),
            'roi': (sum(bets) / len(bets)) * 100,
            'hit_rate': (sum([1 for b in bets if b > 0]) / len(bets)) * 100
        })

odds_df = pd.DataFrame(odds_analysis)
print(odds_df.to_string(index=False))

# -------------------------------------------------------------------------
# 6) HOME vs AWAY TELJESÍTMÉNY
# -------------------------------------------------------------------------

print(f"\n{'='*70}")
print("=== HOME vs AWAY BETTING TELJESÍTMÉNY ===")
print(f"{'='*70}\n")

home_bets = []
away_bets = []

for idx, row in test_results.iterrows():
    if row['pred_prob_home'] >= 1/row['odds_home']:
        profit = (row['odds_home'] - 1) if row['is_home_win'] == 1 else -1
        home_bets.append(profit)
    elif row['pred_prob_away'] >= 1/row['odds_away']:
        profit = (row['odds_away'] - 1) if row['is_home_win'] == 0 else -1
        away_bets.append(profit)

print(f"HOME fogadások:")
print(f"  Bets: {len(home_bets)}")
print(f"  Total Profit: {sum(home_bets):.2f} units")
print(f"  ROI: {(sum(home_bets) / len(home_bets) * 100):.2f}%")
print(f"  Hit Rate: {(sum([1 for b in home_bets if b > 0]) / len(home_bets) * 100):.2f}%")

print(f"\nAWAY fogadások:")
print(f"  Bets: {len(away_bets)}")
print(f"  Total Profit: {sum(away_bets):.2f} units")
print(f"  ROI: {(sum(away_bets) / len(away_bets) * 100):.2f}%")
print(f"  Hit Rate: {(sum([1 for b in away_bets if b > 0]) / len(away_bets) * 100):.2f}%")

# -------------------------------------------------------------------------
# 7) IDŐBELI STABILITÁS & SZEZONALITÁS
# -------------------------------------------------------------------------

print(f"\n{'='*70}")
print("=== IDŐBELI TELJESÍTMÉNY ===")
print(f"{'='*70}\n")

# Rolling profit
test_results = test_results.sort_index()
cumulative_profit = []
rolling_roi = []
window = 20

for idx, row in test_results.iterrows():
    if row['pred_prob_home'] >= 1/row['odds_home']:
        profit = (row['odds_home'] - 1) if row['is_home_win'] == 1 else -1
    elif row['pred_prob_away'] >= 1/row['odds_away']:
        profit = (row['odds_away'] - 1) if row['is_home_win'] == 0 else -1
    else:
        profit = 0
    
    cumulative_profit.append(profit)

test_results['profit'] = cumulative_profit
test_results['cumulative_profit'] = test_results['profit'].cumsum()
test_results['rolling_roi_20'] = test_results['profit'].rolling(window=window).mean() * 100

# Stats
print(f"Időbeli statisztikák (20 meccs rolling window):")
print(f"  Max ROI (rolling): {test_results['rolling_roi_20'].max():.2f}%")
print(f"  Min ROI (rolling): {test_results['rolling_roi_20'].min():.2f}%")
print(f"  Volatility (ROI std): {test_results['rolling_roi_20'].std():.2f}%")

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

test_results['cumulative_profit'].plot(ax=axes[0], linewidth=2, color='blue')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Cumulative Profit (units)')
axes[0].set_title('Cumulative Profit Over Time')
axes[0].grid(True, alpha=0.3)

test_results['rolling_roi_20'].plot(ax=axes[1], linewidth=2, color='green')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('ROI (%)')
axes[1].set_xlabel('Game Index')
axes[1].set_title('Rolling 20-Game ROI')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n{'='*70}")
print("=== ANALÍZIS BEFEJEZVE ===")
print(f"{'='*70}")

In [ ]:
# XGBoost overfitting kezelése

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFECV
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("=== OVERFITTING KEZELÉSE ===")
print("="*70)

# -------------------------------------------------------------------------
# 1) ADATOK ELŐKÉSZÍTÉSE
# -------------------------------------------------------------------------

df = df_enhanced.copy()
df.dropna(subset=["is_home_win"], inplace=True)

df = df.merge(odds[['GAME_ID', 'odds_home', 'odds_away']], 
              left_on='game_id', right_on='GAME_ID', how='left')
df.drop(columns=['GAME_ID'], inplace=True)

y = df["is_home_win"]
feature_cols = [col for col in df.columns if col not in 
                ['is_home_win', 'game_id', 'odds_home', 'odds_away', 
                 'implied_prob_home', 'implied_prob_away', 'book_margin']]
X = df[feature_cols]

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

test_df = df.iloc[split_idx:].copy()

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Features: {X.shape[1]}")

# -------------------------------------------------------------------------
# 2) BASELINE - EREDETI OVERFIT MODELL
# -------------------------------------------------------------------------

print(f"\n{'='*70}")
print("=== 1. BASELINE (Eredeti overfit modell) ===")
print(f"{'='*70}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca_orig = PCA(n_components=0.90)
X_train_pca = pca_orig.fit_transform(X_train_scaled)
X_test_pca = pca_orig.transform(X_test_scaled)

xgb_baseline = XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    min_child_weight=5, gamma=1.0, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
    eval_metric="logloss", random_state=42
)
xgb_baseline.fit(X_train_pca, y_train)

train_proba_base = xgb_baseline.predict_proba(X_train_pca)[:, 1]
test_proba_base = xgb_baseline.predict_proba(X_test_pca)[:, 1]

brier_train_base = brier_score_loss(y_train, train_proba_base)
brier_test_base = brier_score_loss(y_test, test_proba_base)
auc_test_base = roc_auc_score(y_test, test_proba_base)

print(f"PCA komponensek: {pca_orig.n_components_}")
print(f"Brier (Train): {brier_train_base:.4f}")
print(f"Brier (Test):  {brier_test_base:.4f}")
print(f"Delta:         {brier_test_base - brier_train_base:.4f}")
print(f"AUC (Test):    {auc_test_base:.4f}")

# -------------------------------------------------------------------------
# 3) APPROACH 1: KEVESEBB PCA KOMPONENS
# -------------------------------------------------------------------------

print(f"\n{'='*70}")
print("=== 2. KEVESEBB PCA KOMPONENS (5, 8, 10) ===")
print(f"{'='*70}")

pca_results = []

for n_comp in [5, 8, 10, 12]:
    pca_test = PCA(n_components=n_comp)
    X_train_pca_test = pca_test.fit_transform(X_train_scaled)
    X_test_pca_test = pca_test.transform(X_test_scaled)
    
    model = XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        min_child_weight=5, gamma=1.0, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
        eval_metric="logloss", random_state=42
    )
    model.fit(X_train_pca_test, y_train)
    
    train_proba = model.predict_proba(X_train_pca_test)[:, 1]
    test_proba = model.predict_proba(X_test_pca_test)[:, 1]
    
    brier_train = brier_score_loss(y_train, train_proba)
    brier_test = brier_score_loss(y_test, test_proba)
    auc = roc_auc_score(y_test, test_proba)
    
    pca_results.append({
        'n_components': n_comp,
        'brier_train': brier_train,
        'brier_test': brier_test,
        'delta': brier_test - brier_train,
        'auc': auc
    })
    
    print(f"PCA={n_comp}: Brier Train={brier_train:.4f}, Test={brier_test:.4f}, Delta={brier_test-brier_train:.4f}, AUC={auc:.4f}")

# -------------------------------------------------------------------------
# 4) APPROACH 2: ERŐSEBB REGULARIZÁCIÓ (XGBoost params)
# -------------------------------------------------------------------------

print(f"\n{'='*70}")
print("=== 3. ERŐSEBB REGULARIZÁCIÓ (XGBoost) ===")
print(f"{'='*70}")

# Több regularizációs konfiguráció
reg_configs = [
    {"name": "Baseline", "max_depth": 4, "min_child_weight": 5, 
     "gamma": 1.0, "reg_alpha": 0.5, "reg_lambda": 2.0, "n_estimators": 200},
    {"name": "Shallow Trees", "max_depth": 2, "min_child_weight": 10, 
     "gamma": 2.0, "reg_alpha": 1.0, "reg_lambda": 5.0, "n_estimators": 100},
    {"name": "Very Shallow", "max_depth": 2, "min_child_weight": 20, 
     "gamma": 3.0, "reg_alpha": 2.0, "reg_lambda": 10.0, "n_estimators": 50},
    {"name": "Ultra Conservative", "max_depth": 1, "min_child_weight": 30, 
     "gamma": 5.0, "reg_alpha": 5.0, "reg_lambda": 20.0, "n_estimators": 30},
]

# 10 PCA komponens használata (közép)
pca_10 = PCA(n_components=10)
X_train_pca10 = pca_10.fit_transform(X_train_scaled)
X_test_pca10 = pca_10.transform(X_test_scaled)

reg_results = []

for cfg in reg_configs:
    model = XGBClassifier(
        n_estimators=cfg['n_estimators'],
        learning_rate=0.05,
        max_depth=cfg['max_depth'],
        min_child_weight=cfg['min_child_weight'],
        gamma=cfg['gamma'],
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=cfg['reg_alpha'],
        reg_lambda=cfg['reg_lambda'],
        eval_metric="logloss",
        random_state=42
    )
    model.fit(X_train_pca10, y_train)
    
    train_proba = model.predict_proba(X_train_pca10)[:, 1]
    test_proba = model.predict_proba(X_test_pca10)[:, 1]
    
    brier_train = brier_score_loss(y_train, train_proba)
    brier_test = brier_score_loss(y_test, test_proba)
    auc = roc_auc_score(y_test, test_proba)
    
    reg_results.append({
        'config': cfg['name'],
        'brier_train': brier_train,
        'brier_test': brier_test,
        'delta': brier_test - brier_train,
        'auc': auc
    })
    
    print(f"{cfg['name']:20s}: Brier Train={brier_train:.4f}, Test={brier_test:.4f}, Delta={brier_test-brier_train:.4f}, AUC={auc:.4f}")

# -------------------------------------------------------------------------
# 5) APPROACH 3: FEATURE SELECTION (Top K features)
# -------------------------------------------------------------------------

print(f"\n{'='*70}")
print("=== 4. FEATURE SELECTION (Mutual Information) ===")
print(f"{'='*70}")

# Mutual information alapú feature selection
mi_scores = mutual_info_classif(X_train_scaled, y_train, random_state=42)
mi_df = pd.DataFrame({'feature': X.columns, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)

print("\nTop 15 Feature (Mutual Information):")
print(mi_df.head(15).to_string(index=False))

# Különböző feature számokkal
feat_results = []

for k in [5, 10, 15, 20]:
    top_features = mi_df.head(k)['feature'].tolist()
    
    X_train_top = X_train[top_features]
    X_test_top = X_test[top_features]
    
    scaler_k = StandardScaler()
    X_train_top_scaled = scaler_k.fit_transform(X_train_top)
    X_test_top_scaled = scaler_k.transform(X_test_top)
    
    model = XGBClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=2,
        min_child_weight=10, gamma=2.0, subsample=0.7,
        colsample_bytree=0.7, reg_alpha=1.0, reg_lambda=5.0,
        eval_metric="logloss", random_state=42
    )
    model.fit(X_train_top_scaled, y_train)
    
    train_proba = model.predict_proba(X_train_top_scaled)[:, 1]
    test_proba = model.predict_proba(X_test_top_scaled)[:, 1]
    
    brier_train = brier_score_loss(y_train, train_proba)
    brier_test = brier_score_loss(y_test, test_proba)
    auc = roc_auc_score(y_test, test_proba)
    
    feat_results.append({
        'k_features': k,
        'brier_train': brier_train,
        'brier_test': brier_test,
        'delta': brier_test - brier_train,
        'auc': auc
    })
    
    print(f"\nTop {k} features: Brier Train={brier_train:.4f}, Test={brier_test:.4f}, Delta={brier_test-brier_train:.4f}, AUC={auc:.4f}")

# -------------------------------------------------------------------------
# 6) APPROACH 4: EARLY STOPPING
# -------------------------------------------------------------------------

print(f"\n{'='*70}")
print("=== 5. EARLY STOPPING ===")
print(f"{'='*70}")

# Time series split for validation
tscv = TimeSeriesSplit(n_splits=3)
val_idx = list(tscv.split(X_train_pca10))[-1]
train_idx_es, val_idx_es = val_idx

X_train_es = X_train_pca10[train_idx_es]
X_val_es = X_train_pca10[val_idx_es]
y_train_es = y_train.iloc[train_idx_es]
y_val_es = y_train.iloc[val_idx_es]

model_es = XGBClassifier(
    n_estimators=500,  # Magas, de early stopping leállítja
    learning_rate=0.03,
    max_depth=2,
    min_child_weight=10,
    gamma=2.0,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=5.0,
    eval_metric="logloss",
    early_stopping_rounds=20,
    random_state=42
)

model_es.fit(
    X_train_es, y_train_es,
    eval_set=[(X_val_es, y_val_es)],
    verbose=False
)

print(f"Early stopping után használt fák: {model_es.best_iteration}")

# Újratanítás teljes train seten a best_iteration-nel
model_es_final = XGBClassifier(
    n_estimators=model_es.best_iteration,
    learning_rate=0.03,
    max_depth=2,
    min_child_weight=10,
    gamma=2.0,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=5.0,
    eval_metric="logloss",
    random_state=42
)
model_es_final.fit(X_train_pca10, y_train)

train_proba_es = model_es_final.predict_proba(X_train_pca10)[:, 1]
test_proba_es = model_es_final.predict_proba(X_test_pca10)[:, 1]

brier_train_es = brier_score_loss(y_train, train_proba_es)
brier_test_es = brier_score_loss(y_test, test_proba_es)
auc_es = roc_auc_score(y_test, test_proba_es)

print(f"Brier (Train): {brier_train_es:.4f}")
print(f"Brier (Test):  {brier_test_es:.4f}")
print(f"Delta:         {brier_test_es - brier_train_es:.4f}")
print(f"AUC (Test):    {auc_es:.4f}")

# -------------------------------------------------------------------------
# 7) ÖSSZEHASONLÍTÓ TÁBLÁZAT
# -------------------------------------------------------------------------

print(f"\n{'='*70}")
print("=== ÖSSZEHASONLÍTÓ EREDMÉNYEK ===")
print(f"{'='*70}\n")

summary = pd.DataFrame([
    {"Model": "Baseline (17 PCA)", "Brier_Train": brier_train_base, 
     "Brier_Test": brier_test_base, "Delta": brier_test_base - brier_train_base, "AUC": auc_test_base},
    {"Model": "PCA=10", "Brier_Train": pca_results[2]['brier_train'], 
     "Brier_Test": pca_results[2]['brier_test'], "Delta": pca_results[2]['delta'], "AUC": pca_results[2]['auc']},
    {"Model": "Shallow Trees (PCA=10)", "Brier_Train": reg_results[1]['brier_train'], 
     "Brier_Test": reg_results[1]['brier_test'], "Delta": reg_results[1]['delta'], "AUC": reg_results[1]['auc']},
    {"Model": "Very Shallow (PCA=10)", "Brier_Train": reg_results[2]['brier_train'], 
     "Brier_Test": reg_results[2]['brier_test'], "Delta": reg_results[2]['delta'], "AUC": reg_results[2]['auc']},
    {"Model": "Top 10 Features", "Brier_Train": feat_results[1]['brier_train'], 
     "Brier_Test": feat_results[1]['brier_test'], "Delta": feat_results[1]['delta'], "AUC": feat_results[1]['auc']},
    {"Model": "Early Stopping (PCA=10)", "Brier_Train": brier_train_es, 
     "Brier_Test": brier_test_es, "Delta": brier_test_es - brier_train_es, "AUC": auc_es},
])

summary = summary.sort_values('Delta')
print(summary.to_string(index=False))

# Best model
best_model_row = summary.loc[summary['Delta'].abs().idxmin()]
print(f"\n🏆 Legjobb generalizáció (legkisebb delta): {best_model_row['Model']}")
print(f"   Delta: {best_model_row['Delta']:.4f}")
print(f"   AUC: {best_model_row['AUC']:.4f}")

# -------------------------------------------------------------------------
# 8) VIZUALIZÁCIÓ
# -------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Brier scores comparison
x_pos = np.arange(len(summary))
width = 0.35

axes[0].bar(x_pos - width/2, summary['Brier_Train'], width, label='Train', color='skyblue')
axes[0].bar(x_pos + width/2, summary['Brier_Test'], width, label='Test', color='salmon')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(summary['Model'], rotation=45, ha='right')
axes[0].set_ylabel('Brier Score')
axes[0].set_title('Train vs Test Brier Score (Alacsonyabb = Jobb)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Delta comparison
colors = ['green' if d < 0.10 else 'orange' if d < 0.12 else 'red' for d in summary['Delta']]
axes[1].barh(summary['Model'], summary['Delta'], color=colors)
axes[1].axvline(x=0.10, color='green', linestyle='--', alpha=0.7, label='Good (<0.10)')
axes[1].axvline(x=0.15, color='red', linestyle='--', alpha=0.7, label='Bad (>0.15)')
axes[1].set_xlabel('Delta (Test - Train Brier)')
axes[1].set_title('Overfitting Mértéke (Alacsonyabb = Jobb)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n{'='*70}")
print("=== AJÁNLÁS ===")
print(f"{'='*70}")
print("""
Ha a 'Very Shallow' vagy 'Early Stopping' modellnek a legkisebb a delta-ja,
akkor használd azt a konfigurációt a betting stratégiához!

Következő lépés: Futtasd le a betting szimulációt az új, regularizált modellel!
""")

In [ ]:
# Modellek mentése

import os
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# -------------------------------------------------------------------------
# 1) SETUP - Mappa létrehozása
# -------------------------------------------------------------------------

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

# -------------------------------------------------------------------------
# 2) ADATOK ELŐKÉSZÍTÉSE (ugyanaz mint eddig)
# -------------------------------------------------------------------------

df = df_enhanced.copy()
df.dropna(subset=["is_home_win"], inplace=True)

y = df["is_home_win"]
feature_cols = [col for col in df.columns if col not in 
                ['is_home_win', 'game_id', 'odds_home', 'odds_away', 
                 'implied_prob_home', 'implied_prob_away', 'book_margin',
                 'GAME_ID']]
X = df[feature_cols]

# Feature lista mentése
feature_list = X.columns.tolist()

print(f"Features száma: {len(feature_list)}")
print(f"Training samples: {len(X)}")

# -------------------------------------------------------------------------
# 3) KÖZÖS PREPROCESSING - SCALER
# -------------------------------------------------------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------------------------------------------------
# 4) MODELL 1: Random Forest + PCA
# -------------------------------------------------------------------------

print("\n" + "="*50)
print("Training: Random Forest + PCA")
print("="*50)

pca_rf = PCA(n_components=0.90)
X_pca_rf = pca_rf.fit_transform(X_scaled)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42
)
rf_model.fit(X_pca_rf, y)

print(f"PCA komponensek: {pca_rf.n_components_}")
print(f"Variance explained: {pca_rf.explained_variance_ratio_.sum()*100:.2f}%")

# -------------------------------------------------------------------------
# 5) MODELL 2: XGBoost Very Shallow + PCA
# -------------------------------------------------------------------------

print("\n" + "="*50)
print("Training: XGBoost Very Shallow + PCA")
print("="*50)

pca_xgb_shallow = PCA(n_components=10)
X_pca_xgb_shallow = pca_xgb_shallow.fit_transform(X_scaled)

xgb_shallow_model = XGBClassifier(
    n_estimators=50,
    learning_rate=0.05,
    max_depth=2,
    min_child_weight=20,
    gamma=3.0,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=2.0,
    reg_lambda=10.0,
    eval_metric="logloss",
    random_state=42
)
xgb_shallow_model.fit(X_pca_xgb_shallow, y)

print(f"PCA komponensek: {pca_xgb_shallow.n_components_}")

# -------------------------------------------------------------------------
# 6) MODELL 3: XGBoost Tuned + PCA
# -------------------------------------------------------------------------

print("\n" + "="*50)
print("Training: XGBoost Tuned + PCA")
print("="*50)

pca_xgb_tuned = PCA(n_components=0.90)
X_pca_xgb_tuned = pca_xgb_tuned.fit_transform(X_scaled)

xgb_tuned_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    gamma=1.0,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    eval_metric="logloss",
    random_state=42
)
xgb_tuned_model.fit(X_pca_xgb_tuned, y)

print(f"PCA komponensek: {pca_xgb_tuned.n_components_}")

# -------------------------------------------------------------------------
# 7) MENTÉS
# -------------------------------------------------------------------------

print("\n" + "="*50)
print("Mentés folyamatban...")
print("="*50)

# Közös scaler
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.joblib"))

# Feature lista
with open(os.path.join(MODEL_DIR, "feature_columns.json"), 'w') as f:
    json.dump(feature_list, f, indent=2)

# Random Forest + PCA
joblib.dump(rf_model, os.path.join(MODEL_DIR, "rf_pca_model.joblib"))
joblib.dump(pca_rf, os.path.join(MODEL_DIR, "rf_pca_transformer.joblib"))

# XGBoost Very Shallow + PCA
joblib.dump(xgb_shallow_model, os.path.join(MODEL_DIR, "xgb_shallow_model.joblib"))
joblib.dump(pca_xgb_shallow, os.path.join(MODEL_DIR, "xgb_shallow_pca_transformer.joblib"))

# XGBoost Tuned + PCA
joblib.dump(xgb_tuned_model, os.path.join(MODEL_DIR, "xgb_tuned_model.joblib"))
joblib.dump(pca_xgb_tuned, os.path.join(MODEL_DIR, "xgb_tuned_pca_transformer.joblib"))

# Betting config
betting_config = {
    "threshold": 0.53,
    "min_edge": 0.03,
    "odds_range": {"min": 1.5, "max": 2.0},
    "models": {
        "rf_pca": {
            "model_file": "rf_pca_model.joblib",
            "pca_file": "rf_pca_transformer.joblib",
            "description": "Random Forest + PCA (90% variance)"
        },
        "xgb_shallow": {
            "model_file": "xgb_shallow_model.joblib",
            "pca_file": "xgb_shallow_pca_transformer.joblib",
            "description": "XGBoost Very Shallow + PCA (10 components)"
        },
        "xgb_tuned": {
            "model_file": "xgb_tuned_model.joblib",
            "pca_file": "xgb_tuned_pca_transformer.joblib",
            "description": "XGBoost Tuned + PCA (90% variance)"
        }
    }
}

with open(os.path.join(MODEL_DIR, "betting_config.json"), 'w') as f:
    json.dump(betting_config, f, indent=2)

# -------------------------------------------------------------------------
# 8) ÖSSZEGZÉS
# -------------------------------------------------------------------------

print("\n" + "="*50)
print("✅ MENTÉS KÉSZ!")
print("="*50)

print(f"\nMentett fájlok a '{MODEL_DIR}' mappában:")
for fname in os.listdir(MODEL_DIR):
    fpath = os.path.join(MODEL_DIR, fname)
    size = os.path.getsize(fpath) / 1024
    print(f"  📁 {fname} ({size:.1f} KB)")

print(f"\n📋 Feature-ök száma: {len(feature_list)}")
print(f"📊 Modellek száma: 3")
print("""
Használat:
  scaler = joblib.load('models/scaler.joblib')
  pca = joblib.load('models/rf_pca_transformer.joblib')
  model = joblib.load('models/rf_pca_model.joblib')
  
  X_new_scaled = scaler.transform(X_new)
  X_new_pca = pca.transform(X_new_scaled)
  pred_proba = model.predict_proba(X_new_pca)[:, 1]
""")